# NanoForge on a Colab Tesla T4

Choose **Runtime → Change runtime type → T4 GPU** before running. This notebook trains the same GPT implementation used by the local studio, from random weights. It uses CUDA FP16 autocast, gradient scaling, fused AdamW, pinned batch transfers, and uint16 token caches for the bundled 8,192-token Magic BPE tokenizer.

Parameters and optimizer master weights remain FP32 for stability; exported `.pt` tensors are on CPU, so the studio can load them without CUDA. A GPU test below checks training and checkpoint export on your actual Colab hardware.

The example imports Dolly, not the entire multi-hundred-GB catalog. Cosmopedia full imports are unsuitable for a typical Colab disk. Files under `/content` disappear when the runtime is recycled: download checkpoints before disconnecting, or use the optional Drive folder below.


In [ ]:
from pathlib import Path
import os, subprocess, sys
REPO = Path('/content/nanoforge')
if not REPO.exists():
    subprocess.run(['git','clone','https://github.com/bananaosint/nanoforge.git',str(REPO)],check=True)
else:
    subprocess.run(['git','-C',str(REPO),'pull','--ff-only'],check=True)
os.chdir(REPO)
temp = Path('/content/nanoforge-install-tmp'); temp.mkdir(exist_ok=True)
os.environ['TMPDIR'] = str(temp)
os.environ['PIP_NO_CACHE_DIR'] = '1'
# Colab already provides CUDA PyTorch. This installs app dependencies while
# preserving a preinstalled torch version satisfying backend/requirements.txt.
subprocess.run([sys.executable,'-m','pip','install','--no-cache-dir','-r','backend/requirements.txt'],check=True)


In [ ]:
import torch
assert torch.cuda.is_available(), 'Select a GPU runtime, then rerun setup.'
print('GPU:', torch.cuda.get_device_name(0))
print('CUDA capability:', torch.cuda.get_device_capability(0))
print('PyTorch:', torch.__version__)
# T4 has no TF32 support; acceleration comes from FP16 Tensor Cores.
torch.backends.cuda.matmul.allow_tf32 = False
subprocess.run([sys.executable,'-m','pytest','backend/tests','-q'],check=True)


## Choose a run folder

The default uses Colab's temporary local disk. To keep data and checkpoints on Drive instead, uncomment the Drive mount and run-folder lines. Drive can be slower for random token-cache reads. Existing checkpoints can also be uploaded from your computer for resume.


In [ ]:
RUN_HOME = Path('/content/nanoforge-run')
# from google.colab import drive
# drive.mount('/content/drive')
# RUN_HOME = Path('/content/drive/MyDrive/NanoForgeRun')
RUN_HOME.mkdir(parents=True,exist_ok=True)


## Import and pre-tokenize a manageable source

Dolly's full training split is small enough for this example. Source prompts and responses become paired training text. The cache command prints `uint16`; token IDs are expanded to int64 only for the small batches sent to CUDA. The model is still initialized from scratch.


In [ ]:
subprocess.run([sys.executable,'-m','backend.importers','--dataset','dolly','--home',str(RUN_HOME)],check=True)
subprocess.run([sys.executable,'-m','backend.pretokenize','--home',str(RUN_HOME),'--tokenizer','magic8k','--dataset','dolly.jsonl'],check=True)


## Train with FP16 on T4

The Mini profile uses batch 32, context 256, and two accumulation steps (effective batch 64). If Colab reports CUDA out of memory, lower `batch_size` to 16 or 8 and increase accumulation if desired. Leave `precision` as `float16`. Native PyTorch chooses a supported attention implementation; FlashAttention is not forced on the T4.

To resume, set `RESUME` to a full `.pt` checkpoint and keep its tokenizer and architecture. `max_steps` is the number of additional optimizer steps. Change `mode` to `sft` for completion-masked fine-tuning on imported prompt/answer pairs after pretraining.


In [ ]:
import json
config = json.loads((REPO/'configs/colab-t4.json').read_text())
# Optional larger model: fewer than 1 billion parameters, width divisible by heads.
# config.update(preset='Custom', model=dict(n_layer=12, n_head=8, n_embd=768, block_size=256, dropout=0.1))
# Larger models need smaller batches; FP16 does not guarantee they fit a T4.
# config['batch_size'] = 4
# config['max_steps'] = 100
# config['batch_size'] = 16
# config['mode'] = 'sft'
config_file = RUN_HOME/'training-config.json'
config_file.write_text(json.dumps(config,indent=2))
RESUME = None  # Example: RUN_HOME/'checkpoints/pretrain-...-s1000.pt'
command = [sys.executable,'-m','backend.train_cli','--home',str(RUN_HOME),'--config',str(config_file),'--device','cuda']
if RESUME is not None:
    command += ['--resume',str(RESUME)]
subprocess.run(command,check=True)


## Download and open the checkpoint in the studio

The `.pt` includes the exact tokenizer vocabulary, architecture, model weights, optimizer state, step, and precision/scaler metadata. In the updated studio choose **Train & Monitor → Import .pt checkpoint**, then generate. CPU inference uses FP32 automatically. Select your locally available datasets before continuing training.

Alternatively download the ZIP and copy both its `.pt` and `.json` into your local `checkpoints/` folder with the app stopped. The JSON sidecar makes it visible in the selector; the `.pt` upload control can rebuild that metadata itself.


In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files
checkpoints = sorted((RUN_HOME/'checkpoints').glob('*.pt'),key=lambda p:p.stat().st_mtime)
assert checkpoints, 'No checkpoint was saved; inspect the training output.'
latest = checkpoints[-1]
print('Latest checkpoint:', latest)
# CPU-only, weights-only load verifies the portable serialization.
checkpoint = torch.load(latest,map_location='cpu',weights_only=True)
assert checkpoint['tokenizer_state']['kind'] == 'magic8k'
assert all(value.device.type == 'cpu' and value.dtype == torch.float32 for value in checkpoint['model_state_dict'].values())
archive = RUN_HOME/'nanoforge-colab-checkpoint.zip'
with ZipFile(archive,'w',ZIP_DEFLATED) as output:
    output.write(latest,latest.name)
    output.write(latest.with_suffix('.json'),latest.with_suffix('.json').name)
files.download(str(archive))
# To download only the .pt for the studio upload control:
# files.download(str(latest))
